In [0]:
from torn.api import TornAPI
from DBTools.storage import storage
import json
import datetime as dt

header = {"accept": "application/json", "Authorization": f"ApiKey {dbutils.secrets.get('Personal', 'TornAPI')}"}
base_url = "https://api.torn.com/v2/faction/"


tables = ["attacks"]
table_volumes = {"attacks": "torn/faction/faction_api_files/attacks"}
table_params = {"attacks": {"sort": "ASC"}}

torn_api = TornAPI(key= dbutils.secrets.get('Personal', 'TornAPI'), category="faction")
db_storage = storage(spark)

# faction_attacks = torn_api.get_json("attacks")



In [0]:
for table in tables:
    run = True
    params = table_params[table]
    if table =="attacks":
        if spark.catalog.tableExists("torn.faction.attacks"):
            current_data = spark.read.table("torn.faction.attacks")
            max_date = (
                current_data.select("started").agg({"started": "max"}).collect()
            )
            current_max_date = max_date[0]["max(started)"]
            id_list = [id_num[0] for id_num in current_data.select("id").collect()]
        else:
            current_max_date = 1681484237
            id_list = []
        if (current_max_date<= (dt.datetime.today() + dt.timedelta(days=0)).timestamp()):
            params.update({"from_ts": current_max_date})
        else:
            run = False
    print(run)
    table_param = torn_api.prep_params(**params)
    print(table_param)
    if run:
        fac_json = torn_api.get_json(table,table_param)
        db_storage.store(fac_json, table_volumes[table])



In [0]:
fac_json

In [0]:
faction_attacks

In [0]:
for table in tables:

    torn_api.get_json(table)

In [0]:
%sql
SELECT attacks FROM delta.`/Volumes/torn/faction/faction_api_files/faction_attacks`